# pybigtools smoke test

Exercises the primary API surface of `pybigtools` — a Python binding for the Rust `bigtools` crate that reads/writes BigWig and BigBed files.

In [ ]:
# Example 1: Inspect top-level public API surface.
import pybigtools

public_names = [n for n in dir(pybigtools) if not n.startswith('_')]
print(f"pybigtools exposes {len(public_names)} public names")
for name in public_names:
    obj = getattr(pybigtools, name)
    print(f"  {name}: {type(obj).__name__}")

# The `open` function is the canonical entry-point for reading/writing BigWig/BigBed.
assert hasattr(pybigtools, 'open'), "pybigtools.open should exist"

In [ ]:
# Example 2: Write a tiny BigWig file and read it back.
import os
import tempfile

import pybigtools

tmpdir = tempfile.mkdtemp()
bw_path = os.path.join(tmpdir, 'tiny.bw')

# Open in write mode; chromosome sizes are required up front.
chroms = {'chr1': 1000, 'chr2': 500}
writer = pybigtools.open(bw_path, 'w')
# Entries: (chrom, start, end, value)
entries = iter([
    ('chr1', 0, 100, 1.0),
    ('chr1', 100, 200, 2.5),
    ('chr2', 0, 50, 5.0),
])
writer.write(chroms, entries)
del writer

assert os.path.getsize(bw_path) > 0
print(f"Wrote BigWig to {bw_path} ({os.path.getsize(bw_path)} bytes)")

In [ ]:
# Example 3: Read a BigWig file's chrom listing and basic metadata.
import os
import tempfile

import pybigtools

# Re-create the tiny BigWig from Example 2 for self-containment.
tmpdir = tempfile.mkdtemp()
bw_path = os.path.join(tmpdir, 'meta.bw')
chroms = {'chr1': 1000, 'chr2': 500}
writer = pybigtools.open(bw_path, 'w')
writer.write(chroms, iter([('chr1', 0, 100, 1.0), ('chr2', 0, 50, 2.0)]))
del writer

reader = pybigtools.open(bw_path, 'r')
chrom_listing = reader.chroms()
print(f"chroms: {chrom_listing}")

assert 'chr1' in chrom_listing
assert chrom_listing['chr1'] == 1000

In [ ]:
# Example 4: Query interval records from a BigWig file via `records()`.
import os
import tempfile

import pybigtools

tmpdir = tempfile.mkdtemp()
bw_path = os.path.join(tmpdir, 'intervals.bw')
writer = pybigtools.open(bw_path, 'w')
writer.write(
    {'chr1': 1000},
    iter([('chr1', 0, 100, 1.0), ('chr1', 200, 300, 3.0), ('chr1', 500, 600, 7.0)]),
)
del writer

reader = pybigtools.open(bw_path, 'r')
# records() yields (start, end, value) tuples for the queried region.
records = list(reader.records('chr1', 0, 1000))
print(f"Got {len(records)} interval records on chr1")
for rec in records:
    print(f"  {rec}")

assert len(records) == 3
assert records[0] == (0, 100, 1.0)
assert records[-1] == (500, 600, 7.0)

In [ ]:
# Example 5: Get values as a numpy array via `values()`.
import os
import tempfile

import numpy as np
import pybigtools

tmpdir = tempfile.mkdtemp()
bw_path = os.path.join(tmpdir, 'values.bw')
writer = pybigtools.open(bw_path, 'w')
writer.write(
    {'chr1': 100},
    iter([('chr1', 0, 10, 1.0), ('chr1', 10, 20, 2.0), ('chr1', 20, 30, 3.0)]),
)
del writer

reader = pybigtools.open(bw_path, 'r')
values = reader.values('chr1', 0, 30)
print(f"values dtype: {values.dtype}, shape: {values.shape}")
print(f"first 30 values: {values}")

assert isinstance(values, np.ndarray)
assert values.shape == (30,)
assert values[0] == 1.0 and values[15] == 2.0 and values[25] == 3.0
print("\npybigtools smoke test passed.")